# 04. 텍스트 분류 (TF-IDF 베이스라인)

한국어 텍스트이면 **글자 n-gram TF-IDF**가 형태소 분석기 없이도 강력한 베이스라인입니다.

In [ ]:
import sys; sys.path.append("../common")
from utils import *
import numpy as np, pandas as pd
seed_everything()

In [ ]:
DATA_PATH = "../data/text.csv"
TEXT_COL  = "text"
TARGET    = "label"
OUT_PATH  = "../outputs/submission.csv"

In [ ]:
import os
if os.path.exists(DATA_PATH):
    df = read_csv_auto(DATA_PATH)
else:
    print("⚠ 데이터 없음 → 합성 데모 텍스트 사용")
    pos = ["정상 작동 확인 완료", "점검 결과 이상 없음", "장비 상태 양호"]
    neg = ["고장 발생 긴급 수리 필요", "이상 징후 경고 발생", "장비 오류 정지"]
    rows = [(np.random.choice(pos) + f" {i}", 0) for i in range(300)] + [(np.random.choice(neg) + f" {i}", 1) for i in range(300)]
    df = pd.DataFrame(rows, columns=[TEXT_COL, TARGET]).sample(frac=1, random_state=0).reset_index(drop=True)
quick_eda(df, TARGET)

In [ ]:
from sklearn.pipeline import make_pipeline, make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

vec = make_union(TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), min_df=2, sublinear_tf=True),
                 TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1, sublinear_tf=True))
pipe = make_pipeline(vec, LogisticRegression(max_iter=2000, C=5, class_weight="balanced"))
s = cross_val_score(pipe, df[TEXT_COL].fillna(""), df[TARGET], cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1_macro")
print("f1_macro", s.mean().round(4), "±", s.std().round(4))

In [ ]:
pipe.fit(df[TEXT_COL].fillna(""), df[TARGET])
# test 텍스트에 pipe.predict → save_submission(sub, OUT_PATH)

## 개선 아이디어
- LinearSVC/SGD, 앙상블
- 사전학습 임베딩(sentence-transformers)은 CPU 시간 확인 후 소량 데이터에만
- 클래스 불균형 시 임계값/가중치 조정